# 🚀 Day 01a: Hello FastAPI — Your First API in Minutes

---

## 🎯 What You'll Master Today
- Setting up a FastAPI project from scratch
- Creating endpoints with different HTTP methods
- Auto-generated docs (Swagger UI + ReDoc)
- Async vs sync endpoints
- Project structure for real apps

---

## 🎭 Why FastAPI?

```
╔═══════════════════════════════════════════════════════════════════╗
║  WHY FASTAPI WINS                                                ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  🏎️  FAST — One of the fastest Python frameworks               ║
║        Built on Starlette (ASGI) + Pydantic (validation)        ║
║                                                                   ║
║  📝  AUTO DOCS — Swagger UI at /docs, ReDoc at /redoc          ║
║        Generated from your type hints — zero extra work         ║
║                                                                   ║
║  ✅  TYPE SAFE — Python type hints for validation               ║
║        Wrong type? Automatic 422 error with details             ║
║                                                                   ║
║  ⚡  ASYNC NATIVE — Built for async/await from day one         ║
║        But also works perfectly with sync functions              ║
║                                                                   ║
║  🧪  EASY TESTING — TestClient built-in                        ║
║        Uses requests-like interface                              ║
║                                                                   ║
║  STACK: FastAPI → Starlette → ASGI → Uvicorn                   ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# Installation (run this once)
# ============================================================
# pip install fastapi uvicorn[standard]
# That's it. Two packages. You're ready.

# Or for full stack:
# pip install fastapi uvicorn[standard] sqlalchemy pydantic[email] python-jose[cryptography] passlib[bcrypt]

In [ ]:
# ============================================================
# The Simplest FastAPI App — Write This From Memory
# ============================================================

from fastapi import FastAPI

app = FastAPI(
    title="My API",
    description="Interview prep API",
    version="1.0.0"
)

@app.get("/")
async def root():
    return {"message": "Hello, FastAPI!"}

@app.get("/health")
async def health():
    return {"status": "healthy"}

# Run with: uvicorn main:app --reload
# Docs at: http://localhost:8000/docs
print("App created! In a real file, run with:")
print("  uvicorn main:app --reload")
print("  Open: http://localhost:8000/docs")

In [ ]:
# ============================================================
# CRUD Endpoints — The Complete Pattern
# ============================================================
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional

app = FastAPI()

# ---- Pydantic Models (request/response schemas) ----
class UserCreate(BaseModel):
    name: str
    email: str
    age: Optional[int] = None

class UserResponse(BaseModel):
    id: int
    name: str
    email: str
    age: Optional[int] = None

# ---- In-memory "database" ----
users_db: dict[int, dict] = {}
next_id = 1

# ---- CRUD Endpoints ----

@app.get("/users", response_model=list[UserResponse])
async def list_users():
    """List all users."""
    return list(users_db.values())

@app.post("/users", response_model=UserResponse, status_code=201)
async def create_user(user: UserCreate):
    """Create a new user."""
    global next_id
    new_user = {"id": next_id, **user.model_dump()}
    users_db[next_id] = new_user
    next_id += 1
    return new_user

@app.get("/users/{user_id}", response_model=UserResponse)
async def get_user(user_id: int):
    """Get a user by ID."""
    if user_id not in users_db:
        raise HTTPException(status_code=404, detail="User not found")
    return users_db[user_id]

@app.patch("/users/{user_id}", response_model=UserResponse)
async def update_user(user_id: int, user: UserCreate):
    """Update a user."""
    if user_id not in users_db:
        raise HTTPException(status_code=404, detail="User not found")
    users_db[user_id].update(user.model_dump(exclude_unset=True))
    return users_db[user_id]

@app.delete("/users/{user_id}", status_code=204)
async def delete_user(user_id: int):
    """Delete a user."""
    if user_id not in users_db:
        raise HTTPException(status_code=404, detail="User not found")
    del users_db[user_id]

print("✅ Full CRUD API defined!")
print("Endpoints: GET/POST /users, GET/PATCH/DELETE /users/{id}")

In [ ]:
# ============================================================
# Testing with TestClient — No server needed!
# ============================================================
from fastapi.testclient import TestClient

client = TestClient(app)

# Create a user
response = client.post("/users", json={"name": "Sarthak", "email": "s@mail.com", "age": 25})
print(f"POST /users → {response.status_code}: {response.json()}")

# List users
response = client.get("/users")
print(f"GET /users → {response.status_code}: {response.json()}")

# Get specific user
response = client.get("/users/1")
print(f"GET /users/1 → {response.status_code}: {response.json()}")

# 404 for missing user
response = client.get("/users/999")
print(f"GET /users/999 → {response.status_code}: {response.json()}")

# Delete
response = client.delete("/users/1")
print(f"DELETE /users/1 → {response.status_code}")

In [ ]:
# ============================================================
# Async vs Sync — Both work in FastAPI
# ============================================================

# ASYNC — for I/O-bound work (DB, HTTP calls)
# FastAPI runs this on the async event loop
@app.get("/async-endpoint")
async def async_endpoint():
    # await db.fetch_one(query)  # non-blocking DB call
    return {"type": "async"}

# SYNC — for CPU-bound or sync-only libraries
# FastAPI runs this in a thread pool (won't block the event loop)
@app.get("/sync-endpoint")
def sync_endpoint():
    # result = pandas.read_csv("big_file.csv")  # CPU-bound
    return {"type": "sync"}

# RULE: 
# - Use `async def` when you have `await` calls inside
# - Use `def` (no async) for CPU-bound or sync library calls
# - FastAPI handles both correctly!

print("async def → event loop (for await calls)")
print("def → thread pool (for sync/CPU work)")
print("\n⚠️ DON'T use `async def` with blocking code!")
print("   It blocks the entire event loop.")

## 📁 Project Structure — Production Layout

```
╔═══════════════════════════════════════════════════════════════════╗
║  FASTAPI PROJECT STRUCTURE                                       ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  myproject/                                                       ║
║  ├── app/                                                         ║
║  │   ├── __init__.py                                             ║
║  │   ├── main.py              # FastAPI app + startup            ║
║  │   ├── config.py            # Settings (pydantic-settings)     ║
║  │   ├── database.py          # DB connection + session          ║
║  │   ├── models/              # SQLAlchemy models                ║
║  │   │   ├── __init__.py                                        ║
║  │   │   └── user.py                                             ║
║  │   ├── schemas/             # Pydantic schemas                 ║
║  │   │   ├── __init__.py                                        ║
║  │   │   └── user.py                                             ║
║  │   ├── routers/             # API routes (grouped)             ║
║  │   │   ├── __init__.py                                        ║
║  │   │   ├── users.py                                            ║
║  │   │   └── posts.py                                            ║
║  │   ├── services/            # Business logic                   ║
║  │   │   └── user_service.py                                     ║
║  │   └── dependencies.py      # Shared dependencies             ║
║  ├── tests/                                                       ║
║  │   ├── conftest.py                                             ║
║  │   └── test_users.py                                           ║
║  ├── alembic/                 # DB migrations                    ║
║  ├── requirements.txt                                            ║
║  ├── Dockerfile                                                   ║
║  └── docker-compose.yml                                          ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

In [ ]:
# ============================================================
# Router Pattern — Organizing Endpoints
# ============================================================
from fastapi import APIRouter

# In routers/users.py
router = APIRouter(
    prefix="/users",
    tags=["users"],  # groups in Swagger docs
)

@router.get("/")
async def list_users():
    return [{"id": 1, "name": "Alice"}]

@router.get("/{user_id}")
async def get_user(user_id: int):
    return {"id": user_id, "name": "Alice"}

# In main.py
# app.include_router(router)
# app.include_router(posts_router)  # another router

print("APIRouter lets you split endpoints across files.")
print("Include with app.include_router(router)")
print("Tags group them nicely in Swagger docs!")

---

## 🗺️ FastAPI Basics Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  FASTAPI BASICS QUICK REFERENCE                                  ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  SETUP: pip install fastapi uvicorn[standard]                    ║
║  RUN:   uvicorn main:app --reload                                ║
║  DOCS:  /docs (Swagger), /redoc (ReDoc)                         ║
║                                                                   ║
║  DECORATORS: @app.get/post/put/patch/delete("/path")            ║
║  MODELS: Pydantic BaseModel for request/response schemas        ║
║  ERRORS: raise HTTPException(status_code=404, detail="...")     ║
║  ROUTER: APIRouter(prefix="/users", tags=["users"])             ║
║                                                                   ║
║  ASYNC vs SYNC:                                                   ║
║  async def → event loop (use with await)                        ║
║  def → thread pool (sync/CPU-bound work)                        ║
║                                                                   ║
║  TEST: TestClient(app) — no server needed                       ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | What is FastAPI built on? | Starlette (ASGI web framework) + Pydantic (data validation) |
| 2 | How does FastAPI generate docs? | From Python type hints → OpenAPI spec → Swagger UI |
| 3 | async def vs def in FastAPI? | async = event loop, def = thread pool. Don't mix blocking code with async |
| 4 | How to organize a large FastAPI app? | APIRouter for grouping, separate models/schemas/services |
| 5 | How to test FastAPI? | TestClient (sync) or httpx.AsyncClient (async) |
| 6 | FastAPI vs Flask? | FastAPI: async, auto-docs, Pydantic validation. Flask: simpler, bigger ecosystem |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • FastAPI = Starlette + Pydantic + auto docs           │
## │  • @app.get/post/put/patch/delete for CRUD              │
## │  • Pydantic models validate request/response data       │
## │  • HTTPException for error responses                     │
## │  • APIRouter to split code across files                  │
## │  • TestClient for testing without running server         │
## │  • async def for I/O, def for CPU-bound                 │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Path & Query Parameters** — Deep dive into handling different types of input